# Surface-Aware Volumetric Registration Example

This notebook demonstrates how to run the Surface-Aware Volumetric Registration algorithm on the example files in `resources/`.

The example registers the subject `sub-032144` to the `mebrain` template using:

- source and target anatomical volumes;
- source and target cortical surfaces;
- source and target ribbon/aparc label volumes;
- source and target surface label files.

The outputs are written under `examples/outputs/sub-032144_to_mebrain/`.

## 1. Paths

The notebook is stored in `examples/`, so the project root is the parent directory of the notebook folder.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'examples':
    PROJECT_ROOT = PROJECT_ROOT.parent

RESOURCES = PROJECT_ROOT / 'resources'
OUT_DIR = PROJECT_ROOT / 'examples' / 'outputs' / 'sub-032144_to_mebrain'
OUT_DIR.mkdir(parents=True, exist_ok=True)

src_vol = RESOURCES / 'sub-032144_acpc_brain.nii.gz'
trg_vol = RESOURCES / 'mebrain_04mm_LIA.nii.gz'
src_lbl = RESOURCES / 'sub-032144_ribbon_aparc.nii.gz'
trg_lbl = RESOURCES / 'mebrain_04mm_ribbon_aparc_LIA.nii.gz'
src_surf = RESOURCES / 'sub-032144_combined.surf.gii'
trg_surf = RESOURCES / 'mebrain_combined.surf.gii'
src_cort = RESOURCES / 'sub-032144_combined.label.gii'
trg_cort = RESOURCES / 'mebrain_combined.label.gii'

out_vol = OUT_DIR / 'sub-032144_to_mebrain.nii.gz'
out_lbl = OUT_DIR / 'sub-032144_ribbon_aparc_to_mebrain.nii.gz'
out_surf = OUT_DIR / 'sub-032144_combined_to_mebrain.surf.gii'
out_warp = OUT_DIR / 'sub-032144_to_mebrain_warp_x.nii.gz'
out_inv_warp = OUT_DIR / 'mebrain_to_sub-032144_inv_warp_x.nii.gz'

paths = [src_vol, trg_vol, src_lbl, trg_lbl, src_surf, trg_surf, src_cort, trg_cort]
missing = [p for p in paths if not p.exists()]
if missing:
    raise FileNotFoundError('Missing input files:\n' + '\n'.join(str(p) for p in missing))

print('Project root:', PROJECT_ROOT)
print('Output directory:', OUT_DIR)

## 2. Optional input inspection

This cell checks the NIfTI volume shapes and GIFTI surface sizes before registration.

In [ ]:
import nibabel as nib

for p in [src_vol, trg_vol, src_lbl, trg_lbl]:
    img = nib.load(str(p))
    print(f'{p.name}: shape={img.shape}, voxel_sizes={img.header.get_zooms()[:3]}')

for p in [src_surf, trg_surf]:
    gii = nib.load(str(p))
    verts = gii.agg_data('pointset')
    faces = gii.agg_data('triangle')
    print(f'{p.name}: vertices={verts.shape}, faces={faces.shape}')

for p in [src_cort, trg_cort]:
    gii = nib.load(str(p))
    labels = gii.agg_data()
    print(f'{p.name}: labels={labels.shape}')

## 3. Run registration

This cell runs `register_with_surf.py` with the same exposed registration parameters and default values defined in the script:

- `--scales 4 2 1`
- `--iterations 600 500 400`
- `--learning_rate 0.4`
- `--convergence_eps 1e-12 1e-12 1e-12`

The script currently assumes a CUDA-enabled environment.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, str(PROJECT_ROOT / 'register_with_surf.py'),
    '--src_vol', str(src_vol),
    '--trg_vol', str(trg_vol),
    '--src_lbl', str(src_lbl),
    '--trg_lbl', str(trg_lbl),
    '--src_surf', str(src_surf),
    '--trg_surf', str(trg_surf),
    '--src_cort', str(src_cort),
    '--trg_cort', str(trg_cort),
    '--out_vol', str(out_vol),
    '--out_lbl', str(out_lbl),
    '--out_surf', str(out_surf),
    '--out_warp', str(out_warp),
    '--out_inv_warp', str(out_inv_warp),
    '--scales', '4', '2', '1',
    '--iterations', '600', '500', '400',
    '--learning_rate', '0.4',
    '--convergence_eps', '1e-12', '1e-12', '1e-12',
]

print('Running command:')
print(' '.join(cmd))

subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)

## 4. Check outputs

This cell verifies that the expected files were generated.

In [ ]:
expected_outputs = [out_vol, out_lbl, out_surf, out_warp, out_inv_warp]
for p in expected_outputs:
    print(f'{p.name}: exists={p.exists()}, size_mb={(p.stat().st_size / 1024 / 1024 if p.exists() else 0):.2f}')

## 5. Quick visualization

The following cell displays the central axial slice of the target volume, source volume, and registered source volume.

In [ ]:
import matplotlib.pyplot as plt

src_img = nib.load(str(src_vol)).get_fdata()
trg_img = nib.load(str(trg_vol)).get_fdata()
reg_img = nib.load(str(out_vol)).get_fdata()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
items = [
    ('Source', src_img),
    ('Target', trg_img),
    ('Registered source', reg_img),
]

for ax, (title, data) in zip(axes, items):
    z = data.shape[2] // 2
    ax.imshow(data[:, :, z].T, cmap='gray', origin='lower')
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()

## Notes

- The command above intentionally uses the same exposed registration defaults as `register_with_surf.py`.
- The default MSE surface loss assumes source and target surfaces are vertex-wise corresponding.
- The current script writes minimal warp outputs; extend the saving logic if a full 3-channel deformation field is required.